In [ ]:
import numpy as np
import rasterio
import xml.etree.ElementTree as ET
from rasterio.plot import reshape_as_raster, reshape_as_image
from rasterio.enums import Resampling
from rasterio.warp import reproject
from pathlib import Path

# Step 1. XML에서 방사보정계수와 태양 고도각 찾기
def extract_radiometric_factors(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    sun_elevation = float(root.find(".//MEANSUNEL").text)

    radiometric_factors = {}
    for band in root.findall(".//IMD/*"):
        if band.tag.startswith("BAND_"):
            band_name = band.tag
            abs_cal_factor = float(band.find("ABSCALFACTOR").text)
            radiometric_factors[band_name] = abs_cal_factor

    # 팬크로매틱 밴드 처리 (WorldView 데이터의 Pan 밴드는 보통 'BAND_P'로 명명)
    pan_band_element = root.find(".//IMD/P_BAND")
    if pan_band_element:
        pan_abscal_factor = float(pan_band_element.find("ABSCALFACTOR").text)
        radiometric_factors["BAND_P"] = pan_abscal_factor

    return sun_elevation, radiometric_factors

# Step 2. 방사보정 적용
def apply_radiometric_correction(dn_array, abscal_factor, sun_elevation):
    theta = np.deg2rad(90 - sun_elevation)
    cos_theta = np.cos(theta)

    # 0으로 나누는 오류 방지
    if cos_theta == 0:
        cos_theta = 1e-6

    reflectance = (dn_array * abscal_factor) / cos_theta
    # 반사율은 0~1 사이로 클리핑하여 불필요한 값 제거
    reflectance = np.clip(reflectance, 0, 1)

    return reflectance

# Step 3. TIF 데이터 로드 및 방사 보정 적용
def process_tif_with_correction(tif_file, xml_file):
    print(f"✅ TIF 파일 처리 중: {tif_file}")
    sun_elevation, radiometric_factors = extract_radiometric_factors(xml_file)

    bands_data = {}
    with rasterio.open(tif_file) as src:
        # TIF 파일의 밴드 수와 XML의 밴드 수를 일치시킴
        for i in range(1, src.count + 1):
            band_array = src.read(i).astype(np.float32)

            # WorldView-3 멀티스펙트럴 밴드명 유추
            # TIF 파일의 밴드 순서에 따라 밴드명 매핑 (XML에 Band_1, Band_2...)
            band_name = f"BAND_{i}"
            abscal_factor = radiometric_factors.get(band_name)

            if abscal_factor is None:
                # 팬크로매틱 밴드일 경우
                if src.count == 1:
                    band_name = "BAND_P"
                    abscal_factor = radiometric_factors.get(band_name)
                    if abscal_factor is None:
                        raise ValueError(f"XML 파일에서 '{band_name}' 밴드의 ABSCALFACTOR를 찾을 수 없습니다.")
                else:
                    raise ValueError(f"XML 파일에서 '{band_name}' 밴드의 ABSCALFACTOR를 찾을 수 없습니다.")

            reflectance = apply_radiometric_correction(band_array, abscal_factor, sun_elevation)
            bands_data[band_name] = reflectance

        meta = src.meta

    return bands_data, meta

# Step 4. 멀티스펙트럼을 팬크로매틱 크기로 업샘플링 (GDAL 방식과 동일)
def upsample_multi_to_pan(multi_data, multi_meta, pan_meta):
    print("✅ 멀티스펙트럴 밴드 업샘플링 중...")
    upsampled_data = {}
    for band_name, band_array in multi_data.items():
        upsampled_band = np.zeros((pan_meta["height"], pan_meta["width"]), dtype=np.float32)

        reproject(
            source=band_array,
            destination=upsampled_band,
            src_transform=multi_meta["transform"],
            dst_transform=pan_meta["transform"],
            src_crs=multi_meta["crs"],
            dst_crs=pan_meta["crs"],
            resampling=Resampling.cubic
        )
        upsampled_data[band_name] = upsampled_band
    print("✅ 업샘플링 완료")
    return upsampled_data

# Step 5. 팬샤프닝 (Brovey Transform)
def pansharpen_brovey(pan_band, upsampled_multi_bands):
    print("✅ Brovey 팬샤프닝 적용 중...")
    # WorldView-3 밴드 번호와 이름 매핑
    band_map = {2: "BAND_2", 3: "BAND_3", 5: "BAND_5"}

    # RGB 밴드만 추출하여 Brovey 변환에 사용
    rgb_bands = [upsampled_multi_bands[band_map[b]] for b in sorted(band_map.keys())]

    # RGB 밴드의 합산 (분모에 작은 값 더해 0으로 나누는 오류 방지)
    multi_sum = np.sum(rgb_bands, axis=0) + 1e-6

    # 각 RGB 밴드별 팬샤프닝 적용
    pansharpened_bands = {}
    for b_idx in sorted(band_map.keys()):
        band_name = band_map[b_idx]
        pansharpened_bands[band_name] = upsampled_multi_bands[band_name] * (pan_band / multi_sum)

    print("✅ 팬샤프닝 완료")
    return pansharpened_bands

# Step 6. 이미지 저장
def save_processed_image(output_path, bands_data, meta, is_rgb=False):
    print(f"✅ 이미지 저장 중: {output_path}")

    # 0-1.0 범위의 실수 데이터를 0-255 범위의 정수로 스케일링
    # 시각화를 위한 8-bit 이미지 변환
    if is_rgb:
        # RGB 밴드만 스택
        band_map = {2: "BAND_2", 3: "BAND_3", 5: "BAND_5"}
        rgb_array = np.stack([bands_data[band_map[b]] for b in sorted(band_map.keys())], axis=0)

        # 0~1.0 범위를 0~255로 스케일링
        rgb_array_uint8 = (rgb_array * 255).astype(np.uint8)

        new_meta = meta.copy()
        new_meta.update(count=3, dtype=rasterio.uint8)

        with rasterio.open(output_path, 'w', **new_meta) as dst:
            dst.write(rgb_array_uint8)

    else:
        # 멀티밴드 저장
        new_meta = meta.copy()
        new_meta.update(count=len(bands_data), dtype=rasterio.float32)

        with rasterio.open(output_path, 'w', **new_meta) as dst:
            for i, (band_name, band_array) in enumerate(bands_data.items()):
                dst.write(band_array.astype(rasterio.float32), i + 1)

    print(f"✅ {Path(output_path).name} 저장 완료")


# Step 7. 식생지수 계산 및 저장
def calculate_and_save_vegetation_indices(bands_data, output_dir, meta):
    print("✅ 식생지수 계산 및 저장 중...")
    # NDVI 계산 (NIR-1, Red 밴드)
    # 밴드명은 데이터에 따라 "BAND_7", "BAND_5" 등으로 변경 가능
    nir_band = bands_data.get("BAND_7", None)
    red_band = bands_data.get("BAND_5", None)

    if nir_band is not None and red_band is not None:
        ndvi = (nir_band - red_band) / (nir_band + red_band + 1e-6)
        save_processed_image(f"{output_dir}/NDVI_Pansharpened.tif", {"NDVI": ndvi}, meta)

    # GNDVI 계산 (NIR-1, Green 밴드)
    # 밴드명은 데이터에 따라 "BAND_7", "BAND_3" 등으로 변경 가능
    green_band = bands_data.get("BAND_3", None)

    if nir_band is not None and green_band is not None:
        gndvi = (nir_band - green_band) / (nir_band + green_band + 1e-6)
        save_processed_image(f"{output_dir}/GNDVI_Pansharpened.tif", {"GNDVI": gndvi}, meta)

    print("✅ 식생지수 저장 완료")

# 메인 프로세스
def main(pan_tif, pan_xml, multi_tif, multi_xml, output_dir="산출물"):

    Path(output_dir).mkdir(exist_ok=True)

    # 1. TIF 데이터 로드 및 방사 보정
    pan_bands, pan_meta = process_tif_with_correction(pan_tif, pan_xml)
    multi_bands, multi_meta = process_tif_with_correction(multi_tif, multi_xml)

    # 2. 멀티스펙트럴 RGB 이미지 저장 (원래 해상도)
    save_processed_image(f"{output_dir}/RGB_Multi_Original.tif", multi_bands, multi_meta, is_rgb=True)

    # 3. 멀티스펙트럼 밴드를 팬크로매틱 크기로 업샘플링
    upsampled_multi_bands = upsample_multi_to_pan(multi_bands, multi_meta, pan_meta)

    # 4. 팬샤프닝 적용
    pan_band = list(pan_bands.values())[0] # 딕셔너리에서 팬 밴드 데이터 추출
    pansharpened_bands = pansharpen_brovey(pan_band, upsampled_multi_bands)

    # 5. 팬샤프닝된 RGB 이미지 저장
    save_processed_image(f"{output_dir}/RGB_Pansharpened.tif", pansharpened_bands, pan_meta, is_rgb=True)

    # 6. 식생지수 계산 및 저장
    # 팬샤프닝된 모든 밴드를 사용하여 식생지수 계산
    all_pansharpened_bands = {**pansharpened_bands, **upsampled_multi_bands}
    calculate_and_save_vegetation_indices(all_pansharpened_bands, output_dir, pan_meta)

# 예시 사용: 실제 파일 경로로 교체하세요
# if __name__ == '__main__':
#     pan_tif_file = '21DEC18021445-P2BS-050168218010_01_P001.tif'
#     pan_xml_file = '21DEC18021445-P2BS-050168218010_01_P001.xml'
#     multi_tif_file = '21DEC18021445-M1BS-050168218010_01_P001.tif'
#     multi_xml_file = '21DEC18021445-M1BS-050168218010_01_P001.xml'
#
#     main(pan_tif_file, pan_xml_file, multi_tif_file, multi_xml_file)